# Azure Databricks를 사용하여 데이터 품질 제약 조건 구현 및 관리

## 업계 상황: 보험 청구 처리
**InsureCo**는 자동차, 재산, 생명, 건강 및 책임 부문에 걸쳐 중개 네트워크의 청구를 처리하는 중형 보험사입니다.
브로커 시스템은 다양한 형식과 품질 수준으로 청구 데이터를 제출합니다. 수집 시 엄격한 데이터 품질 제어가 이루어지지 않으면 다운스트림 보험 통계 모델, 사기 탐지 파이프라인 및 규제 보고서에 손상된 기록이 수신되어 적립금 계산이 잘못되고 규정 준수 위반이 발생하게 됩니다.
이 데모에서는 **Lakeflow Spark 선언적 파이프라인** 및 **Delta Lake 제약 조건**이 파이프라인의 모든 계층에서 데이터 품질을 적용하는 방법을 안내합니다.
### 데모 설정
시작하기 전에 설정 기능을 실행하여 `trainer_demo.demo_09`에 보험 데모 데이터를 생성하세요.

In [0]:
from scripts.setup import setup, setup_09

spark.sql("USE CATALOG trainer_demo")
setup_09(spark)

**생성된 테이블:**  
| Table | Description |
|---|---|
| `trainer_demo.demo_09.policies` | Clean reference policy data (300 records) |
| `trainer_demo.demo_09.raw_claims` | Raw claims with intentional data quality issues (~515 records) |
| `trainer_demo.demo_09.raw_policies` | Raw policies for schema drift demo (200 records) |

**파이프라인 파일**은 `Demo/pipelines/09-data-quality/`에 있습니다.  
| File | Demonstrates |
|---|---|
| `01_bronze_claims.py` | Nullability, cardinality, range checks |
| `02_type_checks.py` | Data type validation and quarantine pattern |
| `03_schema_drift.py` | Schema drift detection and error handling |
| `04_pipeline_expectations.py` | Full expectations framework |

**정렬 예시:**  
SQL에서 `ORDER BY`를 사용하여 결과를 정렬할 수 있습니다.  
예시:  
sql
SELECT * FROM trainer_demo.demo_09.raw_claims
ORDER BY claim_amount DESC;

## 유효성 검사 구현

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/implement-manage-data-quality-constraints-unity-catalog.implement-validation-checks.png)

### 데모: 유효성 검사 구현
**시나리오**: InsureCo의 원시 청구는 Null, 중복 및 범위를 벗어난 금액이 있는 중개인으로부터 도착합니다.
**파이프라인 파일**: `Demo/pipelines/09-data-quality/01_bronze_claims.py`
**단계**:
1. 원시 청구 데이터를 탐색하고 품질 문제를 식별합니다.  
   - SQL에서 `ORDER BY`를 사용하여 결과를 정렬할 수 있습니다.  
   예시:  
   sql
   SELECT * FROM trainer_demo.demo_09.raw_claims
   ORDER BY claim_amount DESC;
   
2. `WARN` 작업 표시 — 모든 기록이 통과되고 위반이 기록됩니다.
3. `DROP` 작업 표시 — null Claim_ids 및 policy_ids가 제거됨
4. `RANGE` 수표 표시 - 마이너스 및 극단적인 금액이 삭제됨
5. `FAIL` 작업 표시 - 조정자 없이 승인된 청구에 대한 파이프라인 중지
6. 카디널리티 확인 - 중복 청구 ID 보고서 보기
> **Lakeflow Spark 선언적 파이프라인**을 생성하고 `01_bronze_claims.py`을 소스 파일로 추가합니다.  
> 대상 카탈로그를 `trainer_demo`로, 스키마를 `demo_09`로 설정한 다음 **개발** 모드에서 실행합니다.

In [0]:
%sql
-- Explore raw claims data - understand the quality issues before the pipeline run

USE CATALOG trainer_demo;
USE SCHEMA demo_09;

-- Count quality violations across all validation dimensions
SELECT
  COUNT(*) AS total_records,
  COUNT_IF(claim_id IS NULL) AS null_claim_ids,
  COUNT_IF(policy_id IS NULL) AS null_policy_ids,
  COUNT_IF(claim_amount <= 0) AS negative_or_zero_amounts,
  COUNT_IF(claim_amount > 5000000) AS amounts_over_5m,
  COUNT_IF(claim_date > current_date()) AS future_dated_claims,
  COUNT_IF(claim_type NOT IN ('Auto', 'Property', 'Life', 'Health', 'Liability')) AS invalid_claim_types,
  COUNT_IF(status IS NULL) AS null_statuses,
  COUNT_IF(status = 'Approved' AND adjuster_id IS NULL) AS approved_no_adjuster
FROM trainer_demo.demo_09.raw_claims;

In [0]:
%sql
-- Cardinality check — identify duplicate claim submissions (from the unit)

SELECT
  claim_id,
  COUNT(*) AS submission_count
FROM trainer_demo.demo_09.raw_claims
WHERE claim_id IS NOT NULL
GROUP BY claim_id
HAVING COUNT(*) > 1
ORDER BY submission_count DESC
LIMIT 20;

In [0]:
# After running pipeline 01_bronze_claims.py — compare record counts
# The differences show how many records each validation stage dropped

for table, desc in [
    ("raw_claims",               "Raw input"),
    ("bronze_claims_warn",        "WARN: all records kept, violations logged"),
    ("bronze_claims_nullable",    "DROP: nulls on claim_id / policy_id removed"),
    ("bronze_claims_range_checked","DROP: nulls + range + date violations removed"),
    ("bronze_claims_duplicates",  "Duplicate claim IDs (cardinality report)"),
]:
    try:
        cnt = spark.sql(f"SELECT COUNT(*) FROM trainer_demo.demo_09.{table}").collect()[0][0]
        print(f"  {table:<38} {cnt:>5}  {desc}")
    except:
        print(f"  {table:<38}  N/A  (run the pipeline first)")

In [0]:
%sql
-- Delta Lake table constraints — enforce validation rules outside the pipeline
-- Demonstrates NOT NULL and CHECK constraints from the unit
-- Create a clean target table with constraints
CREATE OR REPLACE TABLE trainer_demo.demo_09.claims_with_constraints (
  claim_id INT NOT NULL,
  policy_id STRING NOT NULL,
  claim_amount DOUBLE,
  claim_date DATE,
  claim_type STRING
  --CONSTRAINT positive_claim_amount CHECK (claim_amount > 0),   // Only PRIMARY KEY and FOREIGN KEY constraints are currently supported.
  --CONSTRAINT max_claim_amount CHECK (claim_amount <= 5000000),
  --CONSTRAINT valid_claim_type CHECK (
  --  claim_type IN ('Auto', 'Property', 'Life', 'Health', 'Liability')
  --)
);

In [0]:
%sql
-- Valid insert — should succeed
INSERT INTO trainer_demo.demo_09.claims_with_constraints
VALUES (1, 'POL-00001', 15000.00, current_date(), 'Auto');

In [0]:
%sql
-- Invalid insert — should fail on NOT NULL

INSERT INTO trainer_demo.demo_09.claims_with_constraints
VALUES (NULL, 'POL-00002', 5000.00, current_date(), 'Auto');

## 데이터 유형 검사 구현

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/implement-manage-data-quality-constraints-unity-catalog.implement-data-type-checks.png)

### 데모: 데이터 유형 검사 구현
**시나리오**: InsureCo는 레거시 브로커 피드에서 원시 문자열로 청구 금액과 날짜를 받습니다. 유형 불일치로 인해 다운스트림 보험 통계 계산이 중단됩니다.
**파이프라인 파일**: `Demo/pipelines/09-data-quality/02_type_checks.py`
**단계**:
1. Delta Lake 스키마 적용 표시 - 호환되지 않는 유형을 삽입해 보세요.
2. `TRY_CAST`을 시연하여 유효한 기록과 유효하지 않은 기록을 분류합니다.
3. 숫자 문자열 열의 유효성을 검사하기 위해 REGEXP CHECK 제약 조건을 추가합니다.
4. 파이프라인을 실행하고 `silver_valid_claims`과 `bronze_quarantine_claims`을 비교합니다.
5. SQL에서 `ORDER BY`를 사용하여 결과를 정렬할 수 있습니다.  
   예시:  
   sql
   SELECT * FROM trainer_demo.demo_09.raw_claims
   ORDER BY claim_amount DESC;

In [0]:
%sql

-- 1. Schema enforcement — Delta Lake blocks type-incompatible writes automatically
INSERT INTO trainer_demo.demo_09.policies
VALUES (
  'POL-BAD-001',
  999,
  'Auto',
  '2025-01-01',
  '2026-01-01',
  'not-a-number',
  50000.0
);

In [0]:
%sql

-- 2. Use TRY_CAST to identify type problems — records where result is NULL are invalid

SELECT
  claim_id,
  claim_amount,
  TRY_CAST(claim_amount AS DECIMAL(12,2)) AS validated_amount,
  CASE
    WHEN TRY_CAST(claim_amount AS DECIMAL(12,2)) IS NULL THEN 'Cannot cast to decimal'
    WHEN claim_amount <= 0 THEN 'Non-positive'
    WHEN claim_amount > 5000000 THEN 'Exceeds limit'
    ELSE 'Valid'
  END AS amount_status,
  claim_date,
  TRY_CAST(claim_date AS DATE) AS validated_date
FROM trainer_demo.demo_09.raw_claims
ORDER BY amount_status
LIMIT 20;

In [0]:
# 4. After running pipeline 02_type_checks.py — compare valid vs quarantined records

for table, desc in [
    ("raw_claims",               "Raw input"),
    ("bronze_type_validated_claims", "DROP: type + value violations removed"),
    ("silver_valid_claims",      "Valid records ready for silver"),
    ("bronze_quarantine_claims", "Quarantined: failed type or value validation"),
]:
    try:
        cnt = spark.sql(f"SELECT COUNT(*) FROM trainer_demo.demo_09.{table}").collect()[0][0]
        print(f"  {table:<40} {cnt:>5}  {desc}")
    except:
        print(f"  {table:<40}   N/A  (run pipeline first)")

In [0]:
%sql

-- Show sample quarantined records
SELECT
  claim_id,
  claim_amount,
  claim_type,
  status,
  quarantine_reason
FROM trainer_demo.demo_09.bronze_quarantine_claims
LIMIT 10;

## 스키마 드리프트 감지 및 관리

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/implement-manage-data-quality-constraints-unity-catalog.detect-manage-schema-drift.png)

### 데모: 스키마 드리프트 감지 및 관리
**시나리오**: InsureCo의 브로커 포털 팀이 정책 피드에 `risk_score` 열을 추가합니다.데이터 엔지니어링 팀에 알리지 않고. 파이프라인은 어떻게 반응합니까?
**파이프라인 파일**: `Demo/pipelines/09-data-quality/03_schema_drift.py`
**단계**:1. `raw_policies`의 현재 스키마 표시2. 스키마 드리프트 시뮬레이션 - `risk_score` 열을 추가하고 새 데이터를 삽입합니다.3. 얼마나 엄격한 스트리밍이 실패하는지 관찰합니다. 복구된 데이터 파이프라인은 계속 실행됩니다.4. 자동 스키마 진화를 위해 `mergeSchema` 표시5. 구조적으로 불완전한 기록에 대한 격리 패턴을 보여줍니다.

In [0]:
%sql

-- 1. Inspect the current schema of raw_policies before drift
DESCRIBE TABLE trainer_demo.demo_09.raw_policies;

In [0]:
%sql

-- 2. Simulate schema drift — broker adds a risk_score column
-- In production this would come from an upstream system change

ALTER TABLE trainer_demo.demo_09.raw_policies
ADD COLUMN risk_score DOUBLE;

INSERT INTO trainer_demo.demo_09.raw_policies
VALUES ('POL-DRIFT-001', 1234, 'Auto', '2025-01-01', '2026-01-01', 1200.00, 50000.00, 0.42);

DESCRIBE TABLE trainer_demo.demo_09.raw_policies;

In [0]:
# 3. Rescue mode — the pipeline captures unexpected columns in _rescued_data
# After re-running 03_schema_drift.py, query the rescued data column

try:
    rescued = spark.sql("""
    SELECT policy_id, premium_amount, _rescued_data
    FROM trainer_demo.demo_09.bronze_policies_rescued
    WHERE _rescued_data IS NOT NULL
    LIMIT 10
    """)
    rescued.show(truncate=False)
except Exception:
    print("bronze_policies_rescued not yet created — run 03_schema_drift.py pipeline first")
    print()
    print("rescue mode: unexpected columns are stored as JSON in _rescued_data")
    print("  → pipeline keeps running without interruption")
    print("  → query _rescued_data to understand what new fields arrived")

In [0]:
# 4. Schema evolution — automatically add new columns to the target table

spark.sql("""
CREATE TABLE IF NOT EXISTS trainer_demo.demo_09.policies_evolved
AS SELECT * FROM trainer_demo.demo_09.raw_policies WHERE 1=0
""")

# Write with mergeSchema=true — new columns are auto-added to target
(spark.table("trainer_demo.demo_09.raw_policies")
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable("trainer_demo.demo_09.policies_evolved"))

print("Schema evolution write complete — risk_score column added automatically")
spark.sql("DESCRIBE TABLE trainer_demo.demo_09.policies_evolved").show(truncate=False)

In [0]:
%sql

-- 5. Quarantine pattern — structural checks on raw_policies
-- Records missing required fields are routed to quarantine

SELECT
  COUNT(*) AS valid_count
FROM trainer_demo.demo_09.raw_policies
WHERE policy_id IS NOT NULL
  AND customer_id IS NOT NULL
  AND start_date IS NOT NULL
  AND end_date IS NOT NULL;

In [0]:
%sql

SELECT
  COUNT(*) AS quarantine_count
FROM trainer_demo.demo_09.raw_policies
WHERE policy_id IS NULL
   OR customer_id IS NULL
   OR start_date IS NULL
   OR end_date IS NULL;

In [0]:
%sql

SELECT
  policy_id,
  customer_id,
  start_date,
  end_date
FROM trainer_demo.demo_09.raw_policies
WHERE policy_id IS NULL
   OR customer_id IS NULL
   OR start_date IS NULL
   OR end_date IS NULL
LIMIT 10;

## 파이프라인 기대치에 따라 데이터 품질 관리

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/implement-manage-data-quality-constraints-unity-catalog.manage-data-quality-pipeline-expectations.png)

### 데모: 파이프라인 기대치를 바탕으로 데이터 품질 관리
**시나리오**: InsureCo의 실버 레이어 파이프라인은 명명된 기대치를 사용하여 모든 청구 유형에 대한 비즈니스 규칙을 강화합니다. 파이프라인 UI는 모든 규칙에 대한 통과/실패 지표를 표시합니다.
**파이프라인 파일**: `Demo/pipelines/09-data-quality/04_pipeline_expectations.py`
**단계**:
1. 세 가지 기대 구성요소인 **이름**, **제약조건**, **작업**을 살펴보세요.
2. 재사용 가능한 기대 사전 표시(`expect_all_or_drop`)
3. `WARN`(모니터), `DROP`(깨끗함), `FAIL`(중요 무결성) 비교
4. 파이프라인을 실행하고 파이프라인 UI에서 **데이터 품질** 측정항목을 확인합니다.

**파이프라인 실행 후 기대 지표 보기**:
> 작업 및 파이프라인 → 파이프라인 선택 → 데이터세트(예: `silver_claims`) 클릭  
> 오른쪽 패널에서 **데이터 품질** 탭을 엽니다.

In [0]:
%sql

-- Show what would happen with WARN action (default) — all records flow through
-- Violations are counted but NOT blocked

SELECT
  COUNT(*) AS total_records,
  COUNT_IF(claim_id IS NULL) AS violations_claim_id_null,
  COUNT_IF(policy_id IS NULL) AS violations_policy_id_null,
  COUNT_IF(adjuster_id IS NULL) AS violations_no_adjuster,
  COUNT_IF(claim_amount <= 0) AS violations_bad_amount,
  COUNT_IF(claim_amount > 5000000) AS violations_amount_over_5m,
  COUNT_IF(status NOT IN ('Submitted', 'Under Review', 'Approved', 'Denied')) AS violations_invalid_status,
  COUNT_IF(claim_type NOT IN ('Auto', 'Property', 'Life', 'Health', 'Liberty')) AS violations_invalid_type
FROM trainer_demo.demo_09.raw_claims;

In [0]:
# Reusable expectation dictionaries — how expect_all_or_drop works
# These mirror CLAIM_INTEGRITY_RULES and CLAIM_AMOUNT_RULES in 04_pipeline_expectations.py

integrity_pass = spark.sql("""
SELECT COUNT(*) FROM trainer_demo.demo_09.raw_claims
WHERE claim_id IS NOT NULL AND policy_id IS NOT NULL AND adjuster_id IS NOT NULL
""").collect()[0][0]

amount_pass = spark.sql("""
SELECT COUNT(*) FROM trainer_demo.demo_09.raw_claims
WHERE claim_amount > 0 AND claim_amount <= 5000000
  AND claim_date <= current_date() AND claim_date >= '2000-01-01'
""").collect()[0][0]

total = spark.sql("SELECT COUNT(*) FROM trainer_demo.demo_09.raw_claims").collect()[0][0]

print(f"Total records:                  {total}")
print(f"Pass CLAIM_INTEGRITY_RULES:     {integrity_pass}  (drop {total - integrity_pass})")
print(f"Pass CLAIM_AMOUNT_RULES:        {amount_pass}  (drop {total - amount_pass})")

In [0]:
# FAIL action — check for approved claims with no adjuster (critical violation)
# These records would stop the silver_approved_claims pipeline

critical = spark.sql("""
SELECT claim_id, policy_id, status, adjuster_id, claim_amount
FROM trainer_demo.demo_09.raw_claims
WHERE status = 'Approved' AND adjuster_id IS NULL
""")

count = critical.count()
print(f"Approved claims with no adjuster (would trigger FAIL action): {count}")
if count > 0:
    critical.show(10, truncate=False)

In [0]:
# After running pipeline 04_pipeline_expectations.py — compare all output tables

tables = [
    ("raw_claims",                   "Raw input"),
    ("silver_claims",                "DROP all INTEGRITY + AMOUNT rules"),
    ("silver_claims_monitored",      "WARN  all records kept, violations logged"),
    ("silver_approved_claims",       "FAIL  on missing adjuster for approved"),
    ("silver_high_value_auto_claims","WARN  high-value auto + business rule"),
]

print(f"{'Table':<42} {'Count':>6}  Description")
print("-" * 80)
for table, desc in tables:
    try:
        cnt = spark.sql(f"SELECT COUNT(*) FROM trainer_demo.demo_09.{table}").collect()[0][0]
        print(f"  {table:<40} {cnt:>6}  {desc}")
    except:
        print(f"  {table:<40}    N/A  (run pipeline first)")

### 파이프라인 UI에서 기대 지표 보기
파이프라인을 실행한 후:

1. 왼쪽 사이드바에서 **작업 및 파이프라인**으로 이동합니다.
2. 이름으로 파이프라인을 선택하세요.
3. 파이프라인 그래프에서 데이터세트(예: `silver_claims`)를 클릭합니다.
4. 오른쪽 패널에서 **데이터 품질** 탭을 엽니다.

각 기대에 대해 다음을 확인할 수 있습니다:
- **이름** — 예: `positive_amount`, `valid_claim_type` (설명적인 이름이 중요합니다!)
- **통과 횟수** — 제약 조건을 충족한 레코드
- **실패 횟수** — 제약 조건을 위반한 레코드
- **취한 조치** — 경고/삭제/실패

잘 명명된 기대치는 모니터링 대시보드를 즉시 실행 가능하게 만듭니다.
지표는 파이프라인 실행 전반에 걸쳐 누적되어 시간 경과에 따른 추세 가시성을 제공합니다.